In [13]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

In [14]:
train = pd.read_csv("../Data/train.csv")
test = pd.read_csv("../Data/test.csv")

X = train.drop(columns=["diagnosed_diabetes"])
y = train["diagnosed_diabetes"]

X_test = test.copy()

In [24]:
categorical_cols = [
    'gender',
    'ethnicity',
    'education_level',
    'income_level',
    'smoking_status',
    'employment_status'
]

# Convert to category dtype (important!)
for col in categorical_cols:
    X[col] = X[col].astype('category')

# Convert test set using training categories
for col in categorical_cols:
    test[col] = test[col].astype('category')

X_test = test.copy()

In [16]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [17]:
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "seed": 42,

        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2),
        "num_leaves": trial.suggest_int("num_leaves", 15, 200),
        "max_depth": trial.suggest_int("max_depth", -1, 12),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 200),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 5.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 5.0),
    }

    oof_preds = np.zeros(len(X))

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        train_set = lgb.Dataset(
            X_train,
            y_train,
            categorical_feature=categorical_cols
        )
        valid_set = lgb.Dataset(
            X_valid,
            y_valid,
            categorical_feature=categorical_cols
        )

        model = lgb.train(
            params,
            train_set,
            valid_sets=[valid_set],
            num_boost_round=5000,
            callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False)]
        )

        oof_preds[valid_idx] = model.predict(
            X_valid,
            num_iteration=model.best_iteration
        )

    auc = roc_auc_score(y, oof_preds)
    return auc

In [18]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)   # increase to 200+ for best results

print("Best AUC:", study.best_value)
print("Best Parameters:", study.best_params)

[I 2025-12-11 09:55:23,815] A new study created in memory with name: no-name-6da6202b-39e3-4a16-b3fc-326720361a3a
[I 2025-12-11 09:58:48,572] Trial 0 finished with value: 0.7271271837453123 and parameters: {'learning_rate': 0.01756479190562979, 'num_leaves': 143, 'max_depth': 11, 'min_data_in_leaf': 75, 'feature_fraction': 0.7712531347897873, 'bagging_fraction': 0.5111890505171303, 'bagging_freq': 1, 'lambda_l1': 3.1828968731888994, 'lambda_l2': 1.440111498786623}. Best is trial 0 with value: 0.7271271837453123.
[I 2025-12-11 09:59:17,139] Trial 1 finished with value: 0.7253422440630792 and parameters: {'learning_rate': 0.17877719846469, 'num_leaves': 123, 'max_depth': 6, 'min_data_in_leaf': 60, 'feature_fraction': 0.8613632100293, 'bagging_fraction': 0.6632184618948477, 'bagging_freq': 1, 'lambda_l1': 2.3846244624105535, 'lambda_l2': 1.9322546909228762}. Best is trial 0 with value: 0.7271271837453123.
[I 2025-12-11 09:59:50,043] Trial 2 finished with value: 0.7249281167120598 and para

Best AUC: 0.7297468293063977
Best Parameters: {'learning_rate': 0.07830843170281113, 'num_leaves': 61, 'max_depth': 3, 'min_data_in_leaf': 143, 'feature_fraction': 0.5752373304288648, 'bagging_fraction': 0.9553656376590535, 'bagging_freq': 3, 'lambda_l1': 3.5936746014775727, 'lambda_l2': 0.5148067799836544}


In [19]:
best_params = study.best_params
best_params.update({
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "verbosity": -1,
    "seed": 42
})

final_train_set = lgb.Dataset(X, y)

final_model = lgb.train(
    best_params,
    final_train_set,
    num_boost_round=study.best_trial.number * 200  # heuristic
)

In [25]:
test_probs = final_model.predict(X_test)

In [26]:
test_probs

array([0.47365327, 0.69111764, 0.78199347, ..., 0.6794436 , 0.65271162,
       0.61216696])

In [27]:
# Save Predictions in the required format
submission = pd.DataFrame({
    "id": test["id"],                # test set IDs
    "diagnosed_diabetes": test_probs  # rename column to match required format
})

submission.to_csv("../Submissions/LightGBM.csv", index=False)
